In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import confusion_matrix

In [ ]:
train = pd.read_csv(r"/kaggle/input/playground-series-s3e22/train.csv")
test = pd.read_csv(r"/kaggle/input/playground-series-s3e22/test.csv")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.describe()

In [ ]:
test.describe()

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
plt.style.use('seaborn-darkgrid')
plt.figure(figsize=(10, 6))
sns.countplot(x='outcome', data=train, palette='viridis')
plt.title('Distribution of Outcomes')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.style.use('seaborn-darkgrid')
plt.figure(figsize=(10, 6))
sns.histplot(train['pulse'], kde=True, color='blue', bins=20)
plt.title('Distribution of Pulse')
plt.xlabel('Pulse')
plt.ylabel('Count')
plt.show()

In [ ]:
categorical_cols = ['surgery', 'age', 'temp_of_extremities', 'pain']
plt.figure(figsize=(12, 10))
for i, col in enumerate(categorical_cols, 1):
    plt.subplot(2, 2, i)
    sns.countplot(x=col, data=train, palette='viridis')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = ['rectal_temp', 'pulse', 'respiratory_rate']
plt.figure(figsize=(12, 6))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(1, 3, i)
    plt.hist(train[col], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='rectal_temp', y='pulse', data=train, hue='pain', palette='muted', alpha=0.8)
plt.title('Relationship between Rectal Temperature and Pulse')
plt.xlabel('Rectal Temperature')
plt.ylabel('Pulse')
plt.legend(title='Pain')
plt.grid(True)
plt.show()

In [ ]:
train_missing_cols = train.columns[train.isnull().any()] 
test_missing_cols = test.columns[test.isnull().any()]

train_missing_cols = list(train_missing_cols) 
test_missing_cols = list(test_missing_cols)

In [ ]:
def replace_missing_with_most_common(data, columns):
    for column in columns:
        col_most_common = data[column].value_counts().index[0]
        data[column] = data[column].replace({np.nan: col_most_common})
    return data

train = replace_missing_with_most_common(train, train_missing_cols)
test = replace_missing_with_most_common(test,test_missing_cols)

In [ ]:
numeric_train = train.select_dtypes(include = [np.number]) 
numeric_test = test.select_dtypes(include = [np.number])

categorical_train = train.select_dtypes(exclude = [np.number])
categorical_test = test.select_dtypes(exclude = [np.number])

In [ ]:
train['surgery'] = train['surgery'].replace({'yes': 1,'no': 0})
train['age'] = train['age'].replace({'adult': 1,'young': 0})
train['surgical_lesion'] = train['surgical_lesion'].replace({'yes': 1,'no': 0})
train['cp_data'] = train['cp_data'].replace({'yes': 1,'no': 0})
train['outcome'] = train['outcome'].replace({'died': 0,'euthanized': 1,'lived': 2})

columns = ['temp_of_extremities','peripheral_pulse','mucous_membrane','capillary_refill_time','pain','peristalsis','abdominal_distention',
'nasogastric_tube','nasogastric_reflux','rectal_exam_feces','abdomen','abdomo_appearance']
train = pd.get_dummies(data = train,columns=columns)

In [ ]:
test['surgery'] = test['surgery'].replace({'yes': 1,'no': 0})
test['age'] = test['age'].replace({'adult': 1,'young': 0})
test['surgical_lesion'] = test['surgical_lesion'].replace({'yes': 1,'no': 0})
test['cp_data'] = test['cp_data'].replace({'yes': 1,'no': 0})

test = pd.get_dummies(data = test,columns = ['temp_of_extremities','peripheral_pulse','mucous_membrane','capillary_refill_time','pain',
'peristalsis','abdominal_distention','nasogastric_tube','nasogastric_reflux','rectal_exam_feces','abdomen','abdomo_appearance'])

In [ ]:
y_train = train['outcome']
X_train_id = train['id']
X_train = train.drop(columns = ['outcome', 'id'])

X_test_id = test['id']
X_test = test.drop(columns = 'id')

In [ ]:
X_train['pain_moderate'] = 0 
X_test['nasogastric_reflux_slight'] = 0
X_test['pain_slight'] = 0
X_test['peristalsis_distend_small'] = 0
X_test['rectal_exam_feces_serosanguious'] = 0
X_test = X_test.reindex(X_train.columns, axis=1)

In [ ]:
train_weights = compute_class_weight(class_weight = 'balanced',classes = np.unique(train['outcome']),y = train['outcome'])
skf = StratifiedKFold(n_splits = 5)

In [ ]:
rs=RobustScaler()
X_train_rs = rs.fit_transform(X_train)
X_test_rs = rs.transform(X_test)

In [ ]:
best_random_lgbm = LGBMClassifier(n_estimators = 50,learning_rate = 0.1,num_leaves = 4,class_weight = 'balanced')
best_random_lgbm.fit(X_train_rs, y_train)

In [ ]:
best_grid_xgb = XGBClassifier(max_depth = 11,max_leaves = 4,n_estimators = 150)
best_grid_xgb.fit(X_train_rs, y_train)

In [ ]:
orig_catboost = CatBoostClassifier(random_state = 21,boosting_type = 'Ordered',verbose = 0)
orig_catboost.fit(X_train_rs, y_train)

In [ ]:
def stack_predict_submit(models, X_train, y_train, X_test, X_test_id):
    stacked_model = StackingClassifier(stack_method = 'predict_proba',estimators = models,cv = 'prefit',n_jobs = -1)
    stacked_model.fit(X_train, y_train)
    stacked_predictions = stacked_model.predict(X_test)
    value_to_replace = {0: 'died', 1: 'euthanized', 2: 'lived'}
    stacked_predictions_worded = np.vectorize(value_to_replace.get)(stacked_predictions)
    submission_combined = np.column_stack((X_test_id, stacked_predictions_worded))
    submission_df = pd.DataFrame(submission_combined, columns = ['id', 'outcome'])
    submission_csv = submission_df.to_csv('submission.csv', index=False)
    return submission_csv

best_random_models_list = [('lgbm', best_random_lgbm),('xgb', best_grid_xgb),('orig_catboost', orig_catboost)]
stack_predict_submit(best_random_models_list,X_train_rs,y_train,X_test_rs, X_test_id)

In [ ]:
# Create a mock-up test set using the test data
X_test_mock = X_test_rs[:100] 
y_test_mock = y_train[:100]  

In [ ]:
y_pred_mock = best_random_lgbm.predict(X_test_mock)
conf_mat = confusion_matrix(y_test_mock, y_pred_mock)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_mat, annot=True, cmap='Blues', fmt='g')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
feature_importances = best_random_lgbm.feature_importances_
feature_names = X_train.columns
sorted_idx = np.argsort(feature_importances)[-10:]  # Get indices of top 10 most important features
plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), feature_importances[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Top 10 Feature Importance')
plt.show()